In [ ]:
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
from torch import nn

from torchvision.datasets import ImageFolder
from torchvision import transforms
from torchvision.models import resnet50, ResNet50_Weights
from torchvision.models import vit_b_16, ViT_B_16_Weights
from torchvision.models import vgg19_bn, VGG19_BN_Weights

from transformers import AutoImageProcessor, AutoModel

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr
import time
from tqdm import tqdm
import pickle
from time import time

import os
from pathlib import Path

from Codes.fig4_mis import PrecomputedEmbeddingMetric, compute_score

device = "cuda" if torch.cuda.is_available() else 'cpu'
RAW_DIR = Path('Data/fig4/raw')
RAW_DIR.mkdir(parents=True, exist_ok=True)
nGPU = torch.cuda.device_count()
print(f"Current device: {device}")

In [ ]:
np.var(np.random.normal(0, 1, 1000) * 2)

# get data

In [ ]:
t0 = time()
IMAGENET_VAL_DIR = Path(os.environ.get('IMAGENET_VAL_DIR', 'Data/fig4/raw/imagenet-val'))
val_data = ImageFolder(IMAGENET_VAL_DIR,
                       transform=transforms.ToTensor())
print(f"Load time: {time() - t0:.2f}s")

In [ ]:
class CustomImageDataset(Dataset):
    def __init__(self, transform=None):
        self.transform = transform

    def __len__(self):
        return len(val_data)

    def __getitem__(self, idx):
        image, target = val_data[idx]
        if self.transform:
            image = self.transform(image)

        return image, target

# get models

In [ ]:
# Step 1: Initialize model with the best available weights
weights = VGG19_BN_Weights.DEFAULT
model_vgg19_bn = vgg19_bn(weights=weights)
model_vgg19_bn.eval()

# # Step 2: Initialize the inference transforms
preprocess_vgg19_bn = weights.transforms()

# # behead
model_vgg19_bn.classifier[-1] = torch.nn.Identity()

In [ ]:
# Step 1: Initialize model with the best available weights
weights = ResNet50_Weights.DEFAULT
model_resnet50 = resnet50(weights=weights)
model_resnet50.eval()

# Step 2: Initialize the inference transforms
preprocess_resnet50 = weights.transforms()

# behead
model_resnet50.fc = torch.nn.Identity()

In [ ]:
# Step 1: Initialize model with the best available weights
weights = ViT_B_16_Weights.DEFAULT
model_vit_b_16 = vit_b_16(weights=weights)
model_vit_b_16.eval()

# Step 2: Initialize the inference transforms
preprocess_vit_b_16 = weights.transforms()

# behead
model_vit_b_16.heads = torch.nn.Identity()

In [ ]:
# dinov2
model_dinov2_small = AutoModel.from_pretrained('facebook/dinov2-small')
model_dinov2_small.eval()
preprocess_dinov2_small = AutoImageProcessor.from_pretrained('facebook/dinov2-small')

def transform_dinov2_small(image):
    return preprocess_dinov2_small(
        images=image, return_tensors='pt', do_rescale=False
    )['pixel_values'][0]

x = next(iter(val_data))[0]
inputs = preprocess_dinov2_small(images=x, return_tensors="pt", do_rescale=False)
outputs = model_dinov2_small(**inputs)
last_hidden_states = outputs.last_hidden_state
last_hidden_states.shape, last_hidden_states.min()

# get activations

In [ ]:
dataset_resnet50 = CustomImageDataset(transform=preprocess_resnet50)
resnet50_loader = DataLoader(dataset_resnet50, batch_size=256, shuffle=False, num_workers=8)

batch = next(iter(resnet50_loader))

batch[0].shape

In [ ]:
dataset_resnet50 = CustomImageDataset(transform=preprocess_resnet50)
resnet50_loader = DataLoader(dataset_resnet50, batch_size=256, shuffle=False, num_workers=8)
model_resnet50.to(device)

resnet50_outputs = []
with torch.no_grad():  # Disable gradient calculation
    for i, (inputs, targets) in enumerate(tqdm(resnet50_loader, desc="Processing")):
        # Move inputs and targets to the device (GPU or CPU)
        inputs = inputs.to(device)

        # Forward pass through the model
        outputs = model_resnet50(inputs)

        # Move the data back to CPU and convert it to NumPy for saving
        resnet50_outputs.append(outputs.cpu().numpy())

resnet50_outputs = np.concatenate(resnet50_outputs, 0)

In [ ]:
np.save(RAW_DIR / 'resnet50_outputs.npy', resnet50_outputs)
resnet50_outputs = np.load(RAW_DIR / 'resnet50_outputs.npy')
resnet50_outputs.shape, resnet50_outputs.min()

In [ ]:
dataset_vit_b_16 = CustomImageDataset(transform=preprocess_vit_b_16)
vit_b_16_loader = DataLoader(dataset_vit_b_16, batch_size=256, shuffle=False, num_workers=8)
model_vit_b_16.to(device)

vit_b_16_outputs = []
with torch.no_grad():  # Disable gradient calculation
    for i, (inputs, targets) in enumerate(tqdm(vit_b_16_loader, desc="Processing")):
        # Move inputs and targets to the device (GPU or CPU)
        inputs = inputs.to(device)

        # Forward pass through the model
        outputs = model_vit_b_16(inputs)

        # Move the data back to CPU and convert it to NumPy for saving
        vit_b_16_outputs.append(outputs.cpu().numpy())

vit_b_16_outputs = np.concatenate(vit_b_16_outputs, 0)

In [ ]:
# VGG19-BN activations are computed and saved in the next cell.

In [ ]:
dataset_vgg19_bn = CustomImageDataset(transform=preprocess_vgg19_bn)
vgg19_bn_loader = DataLoader(dataset_vgg19_bn, batch_size=256, shuffle=False, num_workers=8)
model_vgg19_bn.to(device)

vgg19_bn_outputs = []
with torch.no_grad():  # Disable gradient calculation
    for i, (inputs, targets) in enumerate(tqdm(vgg19_bn_loader, desc="Processing")):
        # Move inputs and targets to the device (GPU or CPU)
        inputs = inputs.to(device)

        # Forward pass through the model
        outputs = model_vgg19_bn(inputs)

        # Move the data back to CPU and convert it to NumPy for saving
        vgg19_bn_outputs.append(outputs.cpu().numpy())

vgg19_bn_outputs = np.concatenate(vgg19_bn_outputs, 0)
np.save(RAW_DIR / 'vgg19_bn_outputs.npy', vgg19_bn_outputs)

In [ ]:
np.save(RAW_DIR / 'vit_b_16_outputs.npy', vit_b_16_outputs)
vit_b_16_outputs = np.load(RAW_DIR / 'vit_b_16_outputs.npy')
vit_b_16_outputs.shape, vit_b_16_outputs.min()

In [ ]:
dataset_dinov2_small = CustomImageDataset(transform=transform_dinov2_small)
dinov2_small_loader = DataLoader(dataset_dinov2_small, batch_size=256, shuffle=False, num_workers=8)
model_dinov2_small.to(device)

dinov2_small_outputs = []
with torch.no_grad():  # Disable gradient calculation
    for i, (inputs, targets) in enumerate(tqdm(dinov2_small_loader, desc="Processing")):
        inputs = inputs.to(device)

        # Forward pass through the model
        outputs = model_dinov2_small(inputs).pooler_output

        # Move the data back to CPU and convert it to NumPy for saving
        dinov2_small_outputs.append(outputs.cpu().numpy())

dinov2_small_outputs = np.concatenate(dinov2_small_outputs, 0)

In [ ]:
np.save(RAW_DIR / 'dinov2_small_outputs.npy', dinov2_small_outputs)
dinov2_small_outputs = np.load(RAW_DIR / 'dinov2_small_outputs.npy')
dinov2_small_outputs.shape, dinov2_small_outputs.min()

# compute MIS

In [ ]:
DREAMSIM_EMBEDDINGS = Path(os.environ.get(
    'DREAMSIM_EMBEDDINGS', 'Data/fig4/raw/imagenet-val-dreamsim.pt'
))
dreamsim_embeddings = torch.load(
    DREAMSIM_EMBEDDINGS,
    weights_only=True, map_location='cpu'
)

dreamsim_embeddings.shape

In [ ]:
codes = {
    'resnet50': resnet50_outputs,
    'vgg19': vgg19_bn_outputs,
    'vit_b_16': vit_b_16_outputs,
    'dinov2_small': dinov2_small_outputs,
}

In [ ]:
dreamsim = PrecomputedEmbeddingMetric(dreamsim_embeddings)

In [ ]:
# MIS
result_mis = {}
for key, code in codes.items():
    print('computing', key)
    result_mis[key] = compute_score(
        inputs=np.empty(50000),
        activations=code,
        metrics={
            'dreamsim': dreamsim,
        },
    )

for k, v in result_mis[key].items():
    print(k, v.shape)

In [ ]:
with open(RAW_DIR / 'vision-mis.pickle', 'wb') as handle:
    pickle.dump(result_mis, handle, protocol=pickle.HIGHEST_PROTOCOL)

with open(RAW_DIR / 'vision-mis.pickle', 'rb') as handle:
    result_mis = pickle.load(handle)

In [ ]:
plt.figure(figsize=(6, 3))

plt.subplot(1, 2, 1)
for key, result in result_mis.items():
    plt.errorbar(
        result['quantiles'],
        result['accuracy_dreamsim'].mean(0),
        result['accuracy_dreamsim'].std(0) / np.sqrt(result['accuracy_dreamsim'].shape[0]),
        label=key,
    )
plt.legend()
plt.grid()
plt.xlabel('quantile')
plt.ylabel('accuracy')
plt.title('MIS')

plt.subplot(1, 2, 2)
plt.boxplot(
    [result['accuracy_dreamsim'].mean(1) for result in result_mis.values()]
)
plt.xticks(
    np.arange(1, len(result_mis) + 1),
    list(result_mis.keys()),
    rotation=90
)
plt.grid()
plt.ylabel('accuracy')
plt.title('MIS')
plt.tight_layout()
plt.show()

In [ ]:
def torch_normalize(a):
    a_ = a - torch.mean(a, dim=0, keepdim=True)
    a_ = a_ / (torch.linalg.norm(a_, dim=0, keepdim=True) + 1e-9)
    return a_

def torch_corr(a, b):
    return torch_normalize(a).T @ torch_normalize(b)

In [ ]:
result_mis.keys()

In [ ]:
keys = ['vgg19', 'resnet50', 'vit_b_16', 'dinov2_small']

plt.figure(figsize=(12, 12))
for i, key_a in enumerate(keys):
    avg_mis = result_mis[key_a]['accuracy_dreamsim'].mean(1)
    for j, key_b in tqdm(enumerate(keys)):
        if i == j:
            continue
        corr = abs(torch_corr(
            torch.tensor(codes[key_a]), 
            torch.tensor(codes[key_b]),
        ))
        plt.subplot(4, 4, i * 4 + j + 1)
        max_corrs = torch.max(corr, dim=1)[0].numpy()
        plt.scatter(avg_mis, max_corrs, s=5)
        plt.grid()
        plt.title('%s X %s\ncorr=%.4f' % (
            key_a, key_b, 
            pearsonr(avg_mis, max_corrs)[0]
        ))

plt.tight_layout()
plt.show()

In [ ]:
keys = ['vgg19', 'resnet50', 'vit_b_16', 'dinov2_small']
colors = plt.get_cmap('tab10').colors

heatmap = np.zeros((4, 4))
heatmap[np.diag_indices_from(heatmap)] = np.nan

plt.figure(figsize=(12, 3))
for i, key_a in enumerate(keys):
    avg_mis = result_mis[key_a]['accuracy_dreamsim'].mean(1)
    for j, key_b in tqdm(enumerate(keys)):
        if i == j:
            continue
        corr = abs(torch_corr(
            torch.tensor(codes[key_a]), 
            torch.tensor(codes[key_b]),
        ))
        plt.subplot(1, 4, i + 1)
        max_corrs = torch.max(corr, dim=1)[0].numpy()
        plt.scatter(
            avg_mis, max_corrs, 
            s=500 / np.sqrt(len(avg_mis)), 
            alpha=.5, 
            color=colors[j],
            label=key_b
       )
        plt.grid()
        plt.title('Source Model: %s' % (key_a))
        heatmap[i, j] = pearsonr(avg_mis, max_corrs)[0]
    plt.legend()
    sns.despine(offset=10)
    plt.xlim(.4, .8)
    plt.ylim(0, .65)
    plt.xlabel('Interpretability [MIS acc.]')
    if i == 0:
        plt.ylabel('Max Matching Unit [corr.]')

plt.tight_layout()
plt.savefig(RAW_DIR / 'vision-scatter-raw.pdf')
plt.show()

In [ ]:
plt.figure(figsize=0.7 * np.array((8, 6)))
sns.heatmap(
    heatmap, annot=True, fmt=".2f",
    cmap='bwr', vmin=-1, vmax=1,
    cbar_kws={'label': 'Correlation', 'shrink': 0.94},
    annot_kws={"weight": "bold", "fontsize": 10, "color": "black"},
    square=True,
)
plt.xticks(np.arange(4) + .5, keys)
plt.yticks(np.arange(4) + .5, keys, rotation=0)
plt.tight_layout()
plt.savefig(RAW_DIR / 'vision-correlations-raw.pdf')
plt.show()